# Baseline gegenpressing detection

Notebook ini membandingkan logistic regression dan XGBoost pada fitur
operasional. Split tetap berbasis pertandingan dari notebook 07. Threshold
dipilih hanya menggunakan validation; test tidak digunakan untuk tuning.

Eksekusi saat anotasi belum penuh adalah **smoke test**, bukan hasil penelitian
final. Ubah `FINAL_RUN = True` hanya setelah quality gate seluruh anotasi lulus.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.baseline_training import (
    baseline_readiness,
    prepare_baseline_dataset,
    run_baseline_training,
)
from src.research_readiness import build_model_manifest

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'baseline_smoke'
FINAL_RUN = False

In [2]:
episodes = pd.read_csv(PROCESSED_DIR / 'episode_manifest.csv')
annotations = pd.read_csv(PROCESSED_DIR / 'gegenpressing_annotation_manifest.csv')
features = pd.read_csv(PROCESSED_DIR / 'gegenpressing_baseline_features.csv')

model_manifest = build_model_manifest(episodes, annotations)
dataset = prepare_baseline_dataset(features, model_manifest)
readiness = baseline_readiness(annotations, dataset)

print('Final-training readiness:')
display(readiness.to_frame())
print('Usable labels:', int(dataset['eligible_for_detection_training'].sum()))
print('Mode:', 'FINAL' if FINAL_RUN else 'SMOKE TEST')

Final-training readiness:


,passed
all_annotations_completed,False
all_labels_valid,True
candidate_id_unique,True
confidence_complete,True
confidence_valid,True
uncertain_have_notes,True
annotator_complete,True
both_classes_in_every_split,True
minimum_per_class_in_every_split,False
uncertain_excluded,True


Usable labels: 26
Mode: SMOKE TEST


In [3]:
metrics, predictions, readiness = run_baseline_training(
    dataset,
    annotations,
    ARTIFACT_DIR,
    final_run=FINAL_RUN,
    random_state=42,
)

display(metrics[[
    'model', 'split', 'samples', 'positive_rate',
    'threshold', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc',
]])
print('Artifacts:', ARTIFACT_DIR)
if not FINAL_RUN:
    print('PERINGATAN: metrik pilot tidak boleh dilaporkan sebagai hasil final.')

,model,split,samples,positive_rate,threshold,precision,recall,f1,roc_auc,pr_auc
0,logistic_regression,train,20,0.850000,0.854088,1.000000,0.411765,0.583333,0.980392,0.996732
1,logistic_regression,validation,3,0.666667,0.854088,1.000000,1.000000,1.000000,1.000000,1.000000
2,logistic_regression,test,3,0.666667,0.854088,0.666667,1.000000,0.800000,0.500000,0.833333
3,xgboost,train,20,0.850000,0.879480,1.000000,0.411765,0.583333,1.000000,1.000000
4,xgboost,validation,3,0.666667,0.879480,0.666667,1.000000,0.800000,0.000000,0.583333
5,xgboost,test,3,0.666667,0.879480,1.000000,0.500000,0.666667,0.500000,0.833333


Artifacts: C:\VAEP-Track\artifacts\baseline_smoke
PERINGATAN: metrik pilot tidak boleh dilaporkan sebagai hasil final.
